In [3]:
"""Assignment 3: Spatial Overlays and Areal Interpolation.
Complete three functions using ArcPy and pathlib.Path. Prepare and project
copies before analysis; preserve inputs and avoid hardcoded paths or values.
Check input paths, geometry types, fields, coordinate systems, and output names.
Use try/except handling to catch and report errors.
Find and download census boundaries, a census/ACS numeric attribute, and roads (or
another dataset with line geometry).
"""
def hawkid():
    return ["karin_ulery", "kulery"]

from pathlib import Path
import arcpy


# defining checks sepreatly so that later functions are easier to read

def _require_exists(dataset):
    """Raise an error if an input dataset does not exist."""
    if not arcpy.Exists(str(dataset)):
        raise ValueError(f"Input does not exist: {dataset}")


def _require_geometry(dataset, expected_type):
    """Check that a feature class has the expected geometry type."""
    _require_exists(dataset)

    desc = arcpy.Describe(str(dataset))

    if not hasattr(desc, "shapeType"):
        raise ValueError(f"Input is not a feature class: {dataset}")

    if desc.shapeType != expected_type:
        raise ValueError(
            f"{dataset} must have {expected_type} geometry, "
            f"but has {desc.shapeType}."
        )


def _get_field(dataset, field_name):
    """
    Return the requested ArcPy Field object.
    Raise an error if the field does not exist.
    """
    _require_exists(dataset)

    for field in arcpy.ListFields(str(dataset)):
        if field.name.lower() == field_name.lower():
            return field

    raise ValueError(
        f"Required field '{field_name}' not found in {dataset}."
    )


def _require_numeric_field(dataset, field_name):
    """Check that a field exists and is numeric."""
    field = _get_field(dataset, field_name)

    numeric_types = {
        "SmallInteger",
        "Integer",
        "Single",
        "Double",
        "BigInteger"
    }

    if field.type not in numeric_types:
        raise ValueError(
            f"Field '{field_name}' must be numeric. "
            f"Found type: {field.type}"
        )

    return field


def _require_id_field(dataset, field_name):
    """Check that a polygon ID is text or integer."""
    field = _get_field(dataset, field_name)

    allowed_types = {
        "String",
        "SmallInteger",
        "Integer",
        "BigInteger"
    }

    if field.type not in allowed_types:
        raise ValueError(
            f"Polygon ID field '{field_name}' must be "
            f"text or integer. Found type: {field.type}"
        )

    return field


def _require_known_crs(dataset):
    """Return the spatial reference after checking that it is known."""
    _require_exists(dataset)

    sr = arcpy.Describe(str(dataset)).spatialReference

    if sr is None or sr.name == "Unknown":
        raise ValueError(
            f"Coordinate system is unknown for {dataset}."
        )

    return sr


def _require_projected_meters(dataset):
    """
    Require a known projected coordinate system
    whose linear unit is meters.
    """
    sr = _require_known_crs(dataset)

    if sr.type != "Projected":
        raise ValueError(
            f"{dataset} must use a projected coordinate system. "
            f"Found: {sr.name}"
        )

    unit = sr.linearUnitName.lower()

    if unit not in {"meter", "metre"}:
        raise ValueError(
            f"{dataset} must use meters. "
            f"Found linear unit: {sr.linearUnitName}"
        )

    return sr


def _require_same_crs(dataset1, dataset2):
    """Check that two inputs use the same spatial reference."""
    sr1 = _require_projected_meters(dataset1)
    sr2 = _require_projected_meters(dataset2)

    if sr1.factoryCode != sr2.factoryCode:
        raise ValueError(
            "Inputs must use the same projected coordinate system. "
            f"Found {sr1.name} and {sr2.name}."
        )


def _require_available_output(output_fc):
    """Raise an error rather than overwrite an existing output."""
    if arcpy.Exists(str(output_fc)):
        raise ValueError(
            f"Output already exists: {output_fc}"
        )


def _require_existing_gdb_for_output(output_fc):
    """Require output_fc to be located directly in an existing .gdb."""
    output_path = Path(output_fc)
    parent = output_path.parent

    if parent.suffix.lower() != ".gdb":
        raise ValueError(
            f"Output must be inside a file geodatabase: {output_fc}"
        )

    if not arcpy.Exists(str(parent)):
        raise ValueError(
            f"Output geodatabase does not exist: {parent}"
        )

    _require_available_output(output_fc)

In [8]:
# Problem 1 (20 points)
def prepare_data(input_fc, output_geodatabase, out_coordinate_system,
                 join_table=None, join_field=None, value_field=None):

    try:
        # Convert paths to pathlib Path objects
        input_fc = Path(input_fc)
        output_geodatabase = Path(output_geodatabase)

        # Check that the input feature class exists
        if not arcpy.Exists(str(input_fc)):
            raise ValueError(
                f"Input feature class does not exist: {input_fc}"
            )

        # Describe the input
        desc = arcpy.Describe(str(input_fc))

        # Make sure the input has geometry
        if not hasattr(desc, "shapeType"):
            raise ValueError(
                "input_fc must be a feature class."
            )

        # Check that the input coordinate system is known
        source_sr = desc.spatialReference

        if source_sr.name == "Unknown":
            raise ValueError(
                f"Input has an unknown coordinate system: {input_fc}"
            )

        # Create the requested output spatial reference
        target_sr = arcpy.SpatialReference(out_coordinate_system)

        # Make sure the output coordinate system is valid
        if target_sr.name == "Unknown":
            raise ValueError(
                f"Invalid output coordinate system: "
                f"{out_coordinate_system}"
            )

        # Output CRS must be projected
        if target_sr.type != "Projected":
            raise ValueError(
                "Output coordinate system must be projected."
            )

        # Output CRS must use meters
        if target_sr.linearUnitName.lower() not in {"meter", "metre"}:
            raise ValueError(
                "Output coordinate system must use meters."
            )

        # Create the output geodatabase if it does not exist
        if not arcpy.Exists(str(output_geodatabase)):
            arcpy.management.CreateFileGDB(
                str(output_geodatabase.parent),
                output_geodatabase.name
            )

        # Keep the base name of the input feature class
        output_name = input_fc.stem
        output_fc = output_geodatabase / output_name

        # Do not overwrite an existing final output
        if arcpy.Exists(str(output_fc)):
            raise ValueError(
                f"Output already exists: {output_fc}"
            )

        # Determine whether a table join was requested
        do_join = (
            join_table is not None
            or join_field is not None
            or value_field is not None
        )

        # Normally the original input will be projected
        source_to_project = str(input_fc)

        if do_join:

            # Require all three join arguments
            if (
                join_table is None
                or join_field is None
                or value_field is None
            ):
                raise ValueError(
                    "join_table, join_field, and value_field "
                    "must all be provided for a join."
                )

            # Check that the join table exists
            if not arcpy.Exists(str(join_table)):
                raise ValueError(
                    f"Join table does not exist: {join_table}"
                )

            # Get field names from the feature class
            input_fields = [
                field.name
                for field in arcpy.ListFields(str(input_fc))
            ]

            # Get field names from the join table
            table_fields = [
                field.name
                for field in arcpy.ListFields(str(join_table))
            ]

            # Check the join field in the feature class
            if join_field not in input_fields:
                raise ValueError(
                    f"Join field '{join_field}' not found "
                    f"in {input_fc}."
                )

            # Check the join field in the table
            if join_field not in table_fields:
                raise ValueError(
                    f"Join field '{join_field}' not found "
                    f"in {join_table}."
                )

            # Check the value field in the table
            if value_field not in table_fields:
                raise ValueError(
                    f"Value field '{value_field}' not found "
                    f"in {join_table}."
                )

            # Make a temporary copy so the original input is preserved
            temp_fc = Path(arcpy.env.scratchGDB) / (
                output_name + "_join_tmp"
            )

            # Remove an old temporary copy if one exists
            if arcpy.Exists(str(temp_fc)):
                arcpy.management.Delete(str(temp_fc))

            arcpy.management.CopyFeatures(
                str(input_fc),
                str(temp_fc)
            )

            # Join the requested value field to the temporary copy
            arcpy.management.JoinField(
                str(temp_fc),
                join_field,
                str(join_table),
                join_field,
                [value_field]
            )

            # Project the joined temporary copy
            source_to_project = str(temp_fc)

        # Project into the output geodatabase
        arcpy.management.Project(
            source_to_project,
            str(output_fc),
            target_sr
        )

        # Return the prepared feature class path
        return str(output_fc)

    except Exception as error:
        print(f"Error preparing data: {error}")
        raise

In [19]:
# Problem 2 (40 points)

def estimatePopulationInZone(
        source_polygons,
        population_field,
        study_zone,
        output_fc):
    """
    Estimate the population inside a study zone using areal interpolation.

    The function:
    1. Validates the inputs.
    2. Saves each source polygon's original area as SRC_KM2.
    3. Intersects the source polygons with the study zone.
    4. Calculates each intersected piece's area as PART_KM2.
    5. Estimates population using:

           EST_POP = population * (PART_KM2 / SRC_KM2)

    6. Returns the total estimated population.

    Inputs must be polygon feature classes in the same projected
    coordinate system, with meters as the linear unit.
    """

    try:
        # ----------------------------------------------------------
        # 1. Validate paths
        # ----------------------------------------------------------
        if not arcpy.Exists(source_polygons):
            raise ValueError(
                f"Source polygons do not exist: {source_polygons}"
            )

        if not arcpy.Exists(study_zone):
            raise ValueError(
                f"Study zone does not exist: {study_zone}"
            )

        if arcpy.Exists(output_fc):
            raise ValueError(
                f"Output already exists: {output_fc}"
            )

        # ----------------------------------------------------------
        # 2. Validate geometry, CRS, and units
        # ----------------------------------------------------------
        source_desc = arcpy.Describe(source_polygons)
        zone_desc = arcpy.Describe(study_zone)

        if source_desc.shapeType != "Polygon":
            raise ValueError(
                "source_polygons must contain polygon geometry."
            )

        if zone_desc.shapeType != "Polygon":
            raise ValueError(
                "study_zone must contain polygon geometry."
            )

        source_sr = source_desc.spatialReference
        zone_sr = zone_desc.spatialReference

        if source_sr.name == "Unknown" or zone_sr.name == "Unknown":
            raise ValueError(
                "Both inputs must have a known coordinate system."
            )

        if source_sr.type != "Projected" or zone_sr.type != "Projected":
            raise ValueError(
                "Both inputs must use a projected coordinate system."
            )

        source_units = source_sr.linearUnitName.lower()
        zone_units = zone_sr.linearUnitName.lower()

        if "meter" not in source_units or "meter" not in zone_units:
            raise ValueError(
                "Both projected coordinate systems must use meters."
            )

        # Compare the complete coordinate-system definitions rather
        # than relying only on their displayed names.
        if source_sr.exportToString() != zone_sr.exportToString():
            raise ValueError(
                "source_polygons and study_zone must use the same CRS."
            )

        # ----------------------------------------------------------
        # 3. Validate the population field
        # ----------------------------------------------------------
        source_fields = {
            field.name.lower(): field
            for field in arcpy.ListFields(source_polygons)
        }

        requested_name = population_field.lower()

        if requested_name not in source_fields:
            raise ValueError(
                f"Population field does not exist: {population_field}"
            )

        population_field = source_fields[requested_name].name
        population_type = source_fields[requested_name].type

        numeric_types = {
            "SmallInteger",
            "Integer",
            "Single",
            "Double",
            "BigInteger"
        }

        if population_type not in numeric_types:
            raise ValueError(
                f"{population_field} must be a numeric field."
            )

        # Avoid overwriting assignment output fields that already
        # exist in the source data.
        reserved_fields = {"src_km2", "part_km2", "est_pop"}

        if reserved_fields.intersection(source_fields):
            raise ValueError(
                "The source already contains SRC_KM2, PART_KM2, "
                "or EST_POP. Use a clean source feature class."
            )

        # ----------------------------------------------------------
        # 4. Make a temporary copy so the original input is unchanged
        # ----------------------------------------------------------
        temp_source = arcpy.CreateUniqueName(
            "source_population_copy",
            arcpy.env.scratchGDB
        )

        arcpy.management.CopyFeatures(
            source_polygons,
            temp_source
        )

        # Save the full source-polygon area before intersecting.
        arcpy.management.AddField(
            temp_source,
            "SRC_KM2",
            "DOUBLE"
        )

        arcpy.management.CalculateGeometryAttributes(
            temp_source,
            [["SRC_KM2", "AREA"]],
            area_unit="SQUARE_KILOMETERS"
        )

        # ----------------------------------------------------------
        # 5. Intersect source polygons with the study zone
        # ----------------------------------------------------------
        arcpy.analysis.Intersect(
            [temp_source, study_zone],
            output_fc,
            "ALL",
            None,
            "INPUT"
        )

        # ----------------------------------------------------------
        # 6. Add and calculate PART_KM2 and EST_POP
        # ----------------------------------------------------------
        arcpy.management.AddField(
            output_fc,
            "PART_KM2",
            "DOUBLE"
        )

        arcpy.management.AddField(
            output_fc,
            "EST_POP",
            "DOUBLE"
        )

        arcpy.management.CalculateGeometryAttributes(
            output_fc,
            [["PART_KM2", "AREA"]],
            area_unit="SQUARE_KILOMETERS"
        )

        total_estimated_population = 0.0

        cursor_fields = [
            population_field,
            "SRC_KM2",
            "PART_KM2",
            "EST_POP"
        ]

        with arcpy.da.UpdateCursor(
                output_fc,
                cursor_fields) as cursor:

            for row in cursor:
                population = row[0]
                source_area = row[1]
                part_area = row[2]

                if population is None:
                    estimated_population = 0.0
                elif source_area is None or source_area <= 0:
                    estimated_population = 0.0
                elif part_area is None or part_area <= 0:
                    estimated_population = 0.0
                else:
                    estimated_population = (
                        float(population)
                        * (part_area / source_area)
                    )

                row[3] = estimated_population
                cursor.updateRow(row)

                total_estimated_population += estimated_population

        return total_estimated_population

    except arcpy.ExecuteError:
        print("ArcPy error while estimating population:")
        print(arcpy.GetMessages(2))
        return None

    except Exception as error:
        print(f"Error estimating population: {error}")
        return None

In [23]:
# Problem 3 (40 points)
def summarizeLineLength(lines, polygons, polygon_id_field, output_fc):
    """
    Compute the total length of lines that intersect each polygon.

    Intersect the lines and polygons, measure intersected line lengths,
    summarize the lengths by polygon ID, and join the totals back to a
    copy of all polygons.

    Parameters
    ----------
    lines : str
        Input polyline feature class.

    polygons : str
        Input polygon feature class.

    polygon_id_field : str
        Field that uniquely identifies each polygon.

    output_fc : str
        Output polygon feature class.

    Returns
    -------
    str
        Path to output_fc. Every polygon receives LINE_KM, with zero
        assigned when no lines intersect the polygon.
    """

    # ---------------------------------------------------------
    # 1. Validate that both inputs exist
    # ---------------------------------------------------------
    if not arcpy.Exists(lines):
        raise ValueError("The lines input does not exist: " + str(lines))

    if not arcpy.Exists(polygons):
        raise ValueError(
            "The polygons input does not exist: " + str(polygons)
        )

    line_desc = arcpy.Describe(lines)
    polygon_desc = arcpy.Describe(polygons)

    # ---------------------------------------------------------
    # 2. Validate geometry types
    # ---------------------------------------------------------
    if line_desc.shapeType != "Polyline":
        raise ValueError("The lines input must contain polyline geometry.")

    if polygon_desc.shapeType != "Polygon":
        raise ValueError(
            "The polygons input must contain polygon geometry."
        )

    # ---------------------------------------------------------
    # 3. Validate the polygon ID field
    # ---------------------------------------------------------
    polygon_fields = [
        field.name for field in arcpy.ListFields(polygons)
    ]

    if polygon_id_field not in polygon_fields:
        raise ValueError(
            "Polygon ID field not found: " + polygon_id_field
        )

    # ---------------------------------------------------------
    # 4. Validate coordinate systems
    # ---------------------------------------------------------
    line_sr = line_desc.spatialReference
    polygon_sr = polygon_desc.spatialReference

    if line_sr.name == "Unknown" or polygon_sr.name == "Unknown":
        raise ValueError(
            "Both inputs must have known coordinate systems."
        )

    if line_sr.type != "Projected" or polygon_sr.type != "Projected":
        raise ValueError(
            "Both inputs must use projected coordinate systems."
        )

    if line_sr.factoryCode != polygon_sr.factoryCode:
        raise ValueError(
            "The line and polygon inputs must use the same CRS."
        )

    if line_sr.linearUnitName.lower() not in (
            "meter", "meters", "metre", "metres"):
        raise ValueError(
            "The shared projected CRS must use meters."
        )

    # ---------------------------------------------------------
    # 5. Create temporary dataset names
    # ---------------------------------------------------------
    scratch_gdb = arcpy.env.scratchGDB

    intersect_fc = arcpy.CreateUniqueName(
        "line_polygon_intersect",
        scratch_gdb
    )

    summary_table = arcpy.CreateUniqueName(
        "line_length_summary",
        scratch_gdb
    )

    # ---------------------------------------------------------
    # 6. Copy all polygons to the final output
    # ---------------------------------------------------------
    if arcpy.Exists(output_fc):
        arcpy.management.Delete(output_fc)

    arcpy.management.CopyFeatures(polygons, output_fc)

    # Add the required output field.
    output_fields = [
        field.name for field in arcpy.ListFields(output_fc)
    ]

    if "LINE_KM" not in output_fields:
        arcpy.management.AddField(
            output_fc,
            "LINE_KM",
            "DOUBLE"
        )

    # Initialize every polygon to zero. Polygons without intersecting
    # lines will retain this value.
    arcpy.management.CalculateField(
        output_fc,
        "LINE_KM",
        "0.0",
        "PYTHON3"
    )

    try:
        # ---------------------------------------------------------
        # 7. Intersect lines with polygons
        #
        # output_type="LINE" ensures that the resulting geometry
        # consists of the portions of lines inside polygons.
        # ---------------------------------------------------------
        arcpy.analysis.Intersect(
            [lines, polygons],
            intersect_fc,
            "ALL",
            output_type="LINE"
        )

        intersect_count = int(
            arcpy.management.GetCount(intersect_fc)[0]
        )

        # If no lines intersect any polygons, all LINE_KM values
        # should remain zero.
        if intersect_count == 0:
            return output_fc

        # ---------------------------------------------------------
        # 8. Add and calculate segment lengths
        #
        # SHAPE@LENGTH uses the projected coordinate-system units.
        # The required CRS uses meters, so divide by 1,000 to obtain
        # kilometers.
        # ---------------------------------------------------------
        arcpy.management.AddField(
            intersect_fc,
            "SEG_KM",
            "DOUBLE"
        )

        with arcpy.da.UpdateCursor(
                intersect_fc,
                ["SHAPE@LENGTH", "SEG_KM"]) as cursor:

            for row in cursor:
                length_m = row[0]
                row[1] = length_m / 1000.0
                cursor.updateRow(row)

        # ---------------------------------------------------------
        # 9. Sum segment lengths for every polygon ID
        # ---------------------------------------------------------
        arcpy.analysis.Statistics(
            intersect_fc,
            summary_table,
            [["SEG_KM", "SUM"]],
            polygon_id_field
        )

        # Statistics creates a field named SUM_SEG_KM.
        arcpy.management.JoinField(
            output_fc,
            polygon_id_field,
            summary_table,
            polygon_id_field,
            ["SUM_SEG_KM"]
        )

        # ---------------------------------------------------------
        # 10. Transfer summarized values into LINE_KM
        #
        # A matching polygon receives its total. Polygons without a
        # matching summary record keep zero.
        # ---------------------------------------------------------
        with arcpy.da.UpdateCursor(
                output_fc,
                ["LINE_KM", "SUM_SEG_KM"]) as cursor:

            for row in cursor:
                if row[1] is None:
                    row[0] = 0.0
                else:
                    row[0] = row[1]

                cursor.updateRow(row)

        # The joined summary field is temporary. The required final
        # result field is LINE_KM.
        arcpy.management.DeleteField(
            output_fc,
            ["SUM_SEG_KM"]
        )

    finally:
        # ---------------------------------------------------------
        # 11. Delete temporary datasets
        # ---------------------------------------------------------
        if arcpy.Exists(intersect_fc):
            arcpy.management.Delete(intersect_fc)

        if arcpy.Exists(summary_table):
            arcpy.management.Delete(summary_table)

    return output_fc

In [ ]:
######################################################################
# MAKE NO CHANGES BEYOND THIS POINT.
######################################################################
if __name__ == "__main__" and hawkid()[1] == "hawkid":
print("### Error: YOU MUST provide your HawkID in the hawkid() function.")